# CSE 251B Blended-Data Pipeline

This notebook prepares blended train shards, trains the nano model with root `val.bin` validation, runs the TA evaluation script, and packages the submission files under `results/<timestamp>` before copying them to Drive.

In [1]:
!nvidia-smi

## 1. Mount Drive And Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'blend_new'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'blend_new'
FORCE_REBUILD_DATASET = True
USE_RSYNC_CATCHUP = True
COPY_WORKERS = 2
COPY_RETRIES = 3

RUN_NAME = 'blend_nano_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RESULTS_DIR = REPO_DIR / 'results' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive dataset:', DRIVE_DATASET_DIR)
print('Drive results:', DRIVE_RESULTS_DIR)
print('Repo:', REPO_DIR)
print('Local dataset:', LOCAL_DATASET_DIR)
print('Local run results:', LOCAL_RESULTS_DIR)
print('Drive run results:', DRIVE_RUN_DIR)

## 2. Prepare Repository

In [14]:
!rm -rf /content/milestone
!git clone -b dataset-improved https://github.com/FufenNan/milestone.git /content/milestone
%cd /content/milestone
!git status --short --branch

LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Install Dependencies

In [7]:
!pip install -q tqdm tiktoken "datasets==2.19.2"

## 4. Copy Or Prepare Training Blend

In [15]:
DATASET_LOG = LOCAL_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(DATASET_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

BLEND_SOURCES = {
    'fineweb': {'dir': 'fineweb_edu', 'prefix': 'fineweb', 'min_shards': 40},
    'wikipedia': {'dir': 'wikipedia', 'prefix': 'wikipedia', 'min_shards': 18},
    'arxiv': {'dir': 'papers_arxiv', 'prefix': 'papers_arxiv', 'min_shards': 13},
    'pg19': {'dir': 'pg19', 'prefix': 'pg19', 'min_shards': 13},
}

def count_shards(path):
    path = Path(path)
    return {
        name: len(sorted((path / spec['dir']).glob(f"{spec['prefix']}_train_*.bin")))
        for name, spec in BLEND_SOURCES.items()
    }

def format_counts(counts):
    return ', '.join(f'{name}={counts.get(name, 0)}' for name in BLEND_SOURCES)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob('*_train_*.bin'))

def has_ready_dataset(path):
    counts = count_shards(path)
    return all(counts[name] >= spec['min_shards'] for name, spec in BLEND_SOURCES.items())

def copy_one_file(src_root, dst_root, path, retries=COPY_RETRIES):
    rel = path.relative_to(src_root)
    target = dst_root / rel
    size = path.stat().st_size
    target.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            if target.exists() and target.stat().st_size == size:
                return 'skipped', size
            shutil.copy2(path, target)
            if target.stat().st_size != size:
                raise IOError(f'size mismatch after copy: expected {size}, got {target.stat().st_size}')
            return 'copied', size
        except Exception as exc:
            last_error = exc
            try:
                if target.exists() and target.stat().st_size != size:
                    target.unlink()
            except Exception:
                pass
            if attempt < retries:
                time.sleep(2 * attempt)
    raise RuntimeError(f'copy failed after {retries} attempts: {path} -> {target}: {last_error}')

def rsync_catchup(src, dst):
    cmd = [
        'rsync', '-rt', '--size-only', '--partial', '--info=progress2',
        f'{src}/', f'{dst}/',
    ]
    log('Trying rsync catch-up: ' + ' '.join(cmd))
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if proc.stdout:
        with open(DATASET_LOG, 'a', encoding='utf-8') as f:
            f.write(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f'rsync catch-up failed with exit code {proc.returncode}')

def sync_dir(src, dst, workers=COPY_WORKERS, strict=True):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        log(f'sync skipped: source does not exist: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    if USE_RSYNC_CATCHUP:
        try:
            rsync_catchup(src, dst)
            log(f'rsync catch-up completed: {src} -> {dst}')
            return
        except Exception as exc:
            log(f'WARNING: rsync catch-up failed, falling back to Python copy: {exc}')
    files = [p for p in src.rglob('*') if p.is_file()]
    copied = skipped = copied_bytes = skipped_bytes = 0
    failures = []
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(copy_one_file, src, dst, path) for path in files]
        for future in as_completed(futures):
            try:
                status, size = future.result()
            except Exception as exc:
                failures.append(str(exc))
                continue
            if status == 'copied':
                copied += 1
                copied_bytes += size
            else:
                skipped += 1
                skipped_bytes += size
    log(f'sync {src} -> {dst}: copied {copied} files ({copied_bytes / 1e9:.2f} GB), skipped {skipped} files ({skipped_bytes / 1e9:.2f} GB)')
    if failures:
        msg = f'sync {src} -> {dst}: {len(failures)} files failed; first failure: {failures[0]}'
        if strict:
            raise RuntimeError(msg)
        log('WARNING: ' + msg)

SHARD_SIZE = 100_000_000
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

drive_counts = count_shards(DRIVE_DATASET_DIR)
log(f'Drive shards: {format_counts(drive_counts)}, size={shard_bytes(DRIVE_DATASET_DIR) / 1e9:.2f} GB')

if has_ready_dataset(DRIVE_DATASET_DIR) and not FORCE_REBUILD_DATASET:
    log('Copying blended shards from Drive to repo data/blend_new...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Preparing blend_new shards from scratch; old blend directory is left untouched.')
    if FORCE_REBUILD_DATASET:
        if LOCAL_DATASET_DIR.exists():
            shutil.rmtree(LOCAL_DATASET_DIR)
        if DRIVE_DATASET_DIR.exists():
            shutil.rmtree(DRIVE_DATASET_DIR)
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    log('Preparing blended shards under repo data/blend_new...')
    for source_name, source_spec in BLEND_SOURCES.items():
        cmd = [
            'python', '-u', 'data/blend.py',
            '--data-root', str(LOCAL_DATASET_DIR),
            '--sources', source_name,
            '--streaming',
            '--shard-size', str(SHARD_SIZE),
            '--max-shards-per-source', str(source_spec['min_shards']),
            '--overwrite',
            '--num-proc', str(NUM_PROC),
        ]
        log('Running: ' + ' '.join(cmd))
        with open(DATASET_LOG, 'a', encoding='utf-8') as log_file:
            proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
            ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'Dataset preparation for {source_name} failed with exit code {ret}. See {DATASET_LOG}')
        log(f'Copying prepared {source_name} shards back to Drive...')
        sync_dir(LOCAL_DATASET_DIR / source_spec['dir'], DRIVE_DATASET_DIR / source_spec['dir'], strict=False)
    log('Copying prepared blended shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR, strict=False)

local_counts = count_shards(LOCAL_DATASET_DIR)
log(f'Local shards: {format_counts(local_counts)}, size={shard_bytes(LOCAL_DATASET_DIR) / 1e9:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR), 'Blended dataset is not ready in repo data/blend_new.'

## 5. Train

In [29]:
TRAIN_STDOUT = LOCAL_RESULTS_DIR / 'train_stdout.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

import importlib.util
spec = importlib.util.spec_from_file_location('train_config', REPO_DIR / 'config' / 'config_pg19_no_pubmed_1000.py')
train_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_config)
print('Optimizer:', getattr(train_config, 'optimizer', 'adamw'))
print('Muon lr:', getattr(train_config, 'muon_lr', None))

cmd = ['python', '-u', 'train.py', '--config', 'config/config_pg19_no_pubmed_1000.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config_1000.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config.py']
# cmd = ['python', '-u', 'train.py', '--config', 'config/config_10000.py']
print('Run name:', RUN_NAME)
print('Command:', ' '.join(cmd))

with open(TRAIN_STDOUT, 'w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}. See {TRAIN_STDOUT}')

## 6. Run TA Evaluation

In [7]:
LOCAL_CHECKPOINT = REPO_DIR / 'checkpoints' / 'checkpoint.pt'
EVAL_DATA = REPO_DIR / 'val.bin'
TA_EVAL_JSON = LOCAL_RESULTS_DIR / 'eval_val.json'

assert LOCAL_CHECKPOINT.exists(), f'Missing checkpoint: {LOCAL_CHECKPOINT}'
assert EVAL_DATA.exists(), f'Missing eval data: {EVAL_DATA}'

cmd = [
    'python', '-u', 'evaluate.py',
    '--model_dir', str(REPO_DIR),
    '--checkpoint_filename', 'checkpoints/checkpoint.pt',
    '--data', str(EVAL_DATA),
    '--device', 'cuda',
    '--batch_size', '1',
    '--output_json', str(TA_EVAL_JSON),
]
print('Command:', ' '.join(cmd))

proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'TA evaluation failed with exit code {ret}.')

## 7. Package Results And Copy To Drive

In [30]:
submission_files = {
    REPO_DIR / 'checkpoints' / 'checkpoint.pt': LOCAL_RESULTS_DIR / 'checkpoint.pt',
    # REPO_DIR / 'config' / 'config.py': LOCAL_RESULTS_DIR / 'config.py',
    # REPO_DIR / 'config' / 'config_10000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'config' / 'config_pg19_no_pubmed_1000.py': LOCAL_RESULTS_DIR / 'config.py',
    REPO_DIR / 'model.py': LOCAL_RESULTS_DIR / 'model.py',
}

optional_logs = [
    REPO_DIR / 'checkpoints' / 'train.log',
    REPO_DIR / 'results' / 'train.log',
    REPO_DIR / 'checkpoints' / 'checkpoint_metadata.pt',
]

for src, dst in submission_files.items():
    assert src.exists(), f'Missing expected file: {src}'
    shutil.copy2(src, dst)

for src in optional_logs:
    if src.exists():
        shutil.copy2(src, LOCAL_RESULTS_DIR / src.name)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RESULTS_DIR, DRIVE_RUN_DIR)

print('Local results:', LOCAL_RESULTS_DIR)
print('Drive results:', DRIVE_RUN_DIR)
print('Packaged files:')
for path in sorted(LOCAL_RESULTS_DIR.iterdir()):
    print(' -', path.name)

# 8. Exit Colab

In [16]:
from google.colab import runtime
runtime.unassign()